# Challenge 3: evaluación final e inferencia

El modelo ya fue seleccionado. Ahora debes abrir el conjunto de test una sola vez, comunicar su desempeño y comprobar que el pipeline puede reutilizarse con datos nuevos.

**Entradas:** `test.csv`, `data_contract.json` y `champion_model.joblib`  
**Salidas:** `test_metrics.json`, `batch_predictions.csv`

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    f1_score,
)

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga del modelo y del conjunto de prueba

In [ ]:
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))
training_metadata = json.loads((ARTIFACTS_DIR / "training_metadata.json").read_text(encoding="utf-8"))
model = joblib.load(ARTIFACTS_DIR / "champion_model.joblib")

test_df = pd.read_csv(PROCESSED_DATA_DIR / "test.csv")
X_test = test_df[contract["features"]]
y_test = test_df[contract["target"]]

print("Modelo:", training_metadata["champion"])
print("Test:", X_test.shape)

## 2. Evaluación final

In [ ]:
# Predicciones sobre test (única apertura del conjunto de prueba)
y_pred = model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)
test_f1_macro = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy test : {test_accuracy:.4f}")
print(f"F1 macro test : {test_f1_macro:.4f}\n")
print(classification_report(y_test, y_pred, zero_division=0))

labels = getattr(model, "classes_", np.unique(y_test))
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, labels=labels, ax=ax, cmap="Blues", xticks_rotation=45
)
ax.set_title("Matriz de confusión - test")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
from IPython.display import Markdown, display

# --- Accuracy de CV tomada de training_metadata.json (busca la clave en tiempo de ejecución) ---
def find_cv_accuracy(meta):
    found = []
    def walk(d, prefix=""):
        for k, v in d.items():
            path = f"{prefix}{k}"
            if isinstance(v, dict):
                walk(v, path + ".")
            elif isinstance(v, (int, float)) and not isinstance(v, bool) and "acc" in k.lower() \
                    and any(t in path.lower() for t in ("cv", "val", "mean")):
                found.append((path, float(v)))
    walk(meta)
    return found

cv_candidates = find_cv_accuracy(training_metadata)
print("Claves candidatas de accuracy CV:", cv_candidates)
if cv_candidates:
    cv_key, cv_accuracy = cv_candidates[0]
else:
    cv_key, cv_accuracy = None, None
    print("AVISO: no se encontró accuracy CV en training_metadata.json. "
          "Claves disponibles:", list(training_metadata.keys()))

# --- Análisis de confusiones ---
cm = confusion_matrix(y_test, y_pred, labels=labels)
off = [(labels[i], labels[j], int(cm[i, j])) for i in range(len(labels))
       for j in range(len(labels)) if i != j and cm[i, j] > 0]
off.sort(key=lambda t: -t[2])
top_conf = off[:3]

report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
per_class_f1 = {str(c): report[str(c)]["f1-score"] for c in labels}
worst_class = min(per_class_f1, key=per_class_f1.get)

GAP_THRESHOLD = 0.05  # SUPUESTO: diferencia CV-test > 5 pts se considera señal de sobreajuste
gap = None if cv_accuracy is None else cv_accuracy - test_accuracy

lines = ["### Análisis automático", ""]
if gap is None:
    lines.append("**¿Coherente con CV?** No se pudo comparar: falta la accuracy CV en los metadatos.")
else:
    coherent = abs(gap) <= GAP_THRESHOLD
    lines.append(f"**¿Coherente con CV?** {'Sí' if coherent else 'No'}: accuracy CV = {cv_accuracy:.3f} vs "
                 f"test = {test_accuracy:.3f} (diferencia = {gap:+.3f}; umbral = {GAP_THRESHOLD}).")
if top_conf:
    conf_txt = "; ".join(f"{a} -> {b} ({n} casos)" for a, b, n in top_conf)
    lines.append(f"**Clases confundidas:** {conf_txt}.")
else:
    lines.append("**Clases confundidas:** no hay errores en test.")
lines.append(f"**Clase más débil (F1):** {worst_class} ({per_class_f1[worst_class]:.3f}).")
if gap is None:
    lines.append("**¿Sobreajuste?** No evaluable sin la métrica CV.")
elif gap > GAP_THRESHOLD:
    lines.append("**¿Sobreajuste?** Hay indicios: test queda claramente por debajo de CV.")
else:
    lines.append("**¿Sobreajuste?** No hay evidencia: test es similar (o mejor) que CV. "
                 "Nota: con test pequeño la estimación tiene incertidumbre.")
lines.append(f"(n test = {len(y_test)})")
display(Markdown("\n\n".join(lines)))

**Análisis:**

> ¿El desempeño es coherente con validación cruzada? ¿Qué clases se confunden? ¿Hay evidencia de sobreajuste?

La celda de código anterior calcula las respuestas con los números reales de tu ejecución. Interpretación:

- **Coherencia con CV:** se compara la accuracy de CV con la de test. Una diferencia pequeña (≤ 5 puntos) indica que la estimación de CV era fiable y que no hubo fuga hacia test.
- **Clases confundidas:** se leen de la matriz de confusión (pares real → predicho con más errores) y de las clases con menor F1 en el `classification_report`.
- **Sobreajuste:** se sospecha si test es claramente peor que CV. Si son similares, no hay evidencia de sobreajuste (ni de selección de modelo contaminada por test).

## 3. Función de inferencia

In [ ]:
feature_columns = contract["features"]

def predict(records: pd.DataFrame) -> pd.DataFrame:
    """Valida el esquema y devuelve las predicciones del pipeline."""
    if not isinstance(records, pd.DataFrame):
        raise TypeError("records debe ser un pandas.DataFrame")
    if records.empty:
        raise ValueError("records no contiene filas")

    # Columnas faltantes -> error explícito (nunca se rellenan en silencio)
    missing = [c for c in feature_columns if c not in records.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas: {missing}")

    # Columnas extra -> se ignoran, pero se avisa
    extra = [c for c in records.columns if c not in feature_columns]
    if extra:
        print(f"AVISO: se ignoran columnas no incluidas en el contrato: {extra}")

    # Mismo orden que en entrenamiento
    X = records[feature_columns].copy()

    out = pd.DataFrame(index=records.index)
    out["prediction"] = model.predict(X)

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        for i, cls in enumerate(model.classes_):
            out[f"proba_{cls}"] = proba[:, i]
    else:
        print("AVISO: el modelo no expone predict_proba; se devuelve solo prediction.")
    return out

## 4. Predicción por lotes

In [ ]:
# Simulación de datos nuevos. En producción no provendrían del test set.
new_records = X_test.iloc[:5].copy()

batch_predictions = predict(new_records)
print(batch_predictions)

# Comprobación de la validación del esquema: debe fallar con una columna faltante
try:
    predict(new_records.drop(columns=[feature_columns[0]]))
except ValueError as e:
    print("Validación OK ->", e)

batch_predictions.to_csv(REPORTS_DIR / "batch_predictions.csv", index=True, index_label="row_id")
print("Guardado:", REPORTS_DIR / "batch_predictions.csv")

## 5. Reporte final

In [ ]:
test_metrics = {
    "model": training_metadata["champion"],
    "accuracy": float(test_accuracy),
    "f1_macro": float(test_f1_macro),
    "n_test": int(len(y_test)),
}
(REPORTS_DIR / "test_metrics.json").write_text(
    json.dumps(test_metrics, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(json.dumps(test_metrics, indent=2, ensure_ascii=False))

In [ ]:
assert (REPORTS_DIR / "test_metrics.json").exists()
assert (REPORTS_DIR / "batch_predictions.csv").exists()
print("Challenge 3 completado.")

In [ ]:
conf_cell = ", ".join(f"{a} -> {b}" for a, b, _ in top_conf) if top_conf else "Sin errores"
cv_cell = f"{cv_accuracy:.3f}" if cv_accuracy is not None else "No disponible en metadatos"
limit_parts = [f"clase más débil: {worst_class} (F1 = {per_class_f1[worst_class]:.2f})",
               f"test de solo {len(y_test)} filas"]
if gap is not None and gap > GAP_THRESHOLD:
    limit_parts.append("posible sobreajuste")

display(Markdown(f"""
## Resumen ejecutivo

| Elemento | Resultado |
|---|---|
| Modelo campeón | {training_metadata["champion"]} |
| Accuracy CV | {cv_cell} |
| Accuracy test | {test_accuracy:.3f} |
| F1 macro test | {test_f1_macro:.3f} |
| Clases más confundidas | {conf_cell} |
| Principal limitación | {"; ".join(limit_parts)} |
"""))

## Checklist

- [x] Evalué únicamente el modelo campeón.
- [x] No reajusté hiperparámetros después de observar test.
- [x] Implementé validación básica del esquema de entrada.
- [x] Generé predicciones por lotes.
- [x] Guardé las métricas finales.